In [1]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import pandas as pd
import torch
from datasets import load_dataset, concatenate_datasets, DatasetDict


device = "cuda" if torch.cuda.is_available() else "cpu"


model_path = "models/whisper_small_mansi_v2"

pred_path = 'inference'


processor = WhisperProcessor.from_pretrained(model_path, language="hungarian", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_path).to(device)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None
model.generation_config._from_model_config = True


/home/dev/workspace/shidlovskiy/mansi/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
tts_path = "../data/tts_dataset/val"
tts_val_dataset = load_dataset('csv', data_files=tts_path+'/metadata.csv', delimiter='\t', split='train')

stt_path = "../data/stt_dataset/val"
stt_val_dataset = load_dataset('csv', data_files=stt_path+'/metadata.csv', delimiter='\t', split='train')

bonus_path = "../data/bonus_dataset/val"
bonus_val_dataset = load_dataset('csv', data_files=bonus_path+'/metadata.csv', delimiter='\t', split='train')

tts_val_dataset = tts_val_dataset.map(
    lambda x: {"audio_file": tts_path + "/wavs/" + x["audio_file"]}
)

stt_val_dataset = stt_val_dataset.map(
    lambda x: {"audio_file": stt_path + "/wavs/" + x["audio_file"]}
)

bonus_val_dataset = bonus_val_dataset.map(
    lambda x: {"audio_file": bonus_path + "/wavs/" + x["audio_file"]}
)


dataset = DatasetDict({
    "val": concatenate_datasets([tts_val_dataset, stt_val_dataset, bonus_val_dataset]),
})

dataset = dataset["val"].shuffle(seed=42).select(range(100)).remove_columns('speaker')
dataset

FileNotFoundError: Unable to find '/home/dev/workspace/shidlovskiy/mansi/stt/../data/bonus_dataset/metadata.csv'

In [ ]:
import os
import shutil
import pandas as pd
import librosa


os.makedirs(pred_path, exist_ok=True)
os.makedirs(f'{pred_path}/wavs', exist_ok=True)

metadata = pd.DataFrame(columns=['reference', 'model_out'])

for sample in dataset:
    audio_path = sample['audio_file']
    filename = os.path.basename(audio_path)
    out_path = os.path.join(pred_path, 'wavs', filename)

    shutil.copy(audio_path, out_path)

    audio_file, sr = librosa.load(audio_path, sr=16000)
    inputs = processor(audio_file, sampling_rate=16000, return_tensors="pt").to(device)
    pred_ids = model.generate(inputs.input_features, max_length=225)
    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]

    new_row = pd.DataFrame([{
        'reference': filename,
        'model_out': pred_text
    }])
    metadata = pd.concat([metadata, new_row], ignore_index=True)

metadata.to_csv(f'{pred_path}/metadata.csv', index=False, sep="\t")

In [ ]:
!zip -r stt_inference.zip inference